trains Linear Regression, Random Forest, and XGBoost on:

1. Non-PCA features
2. PCA features

## 1. Imports and paths

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed. Install using: pip install xgboost")

PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

NO_DR_DIR = PROJECT_DIR / "model_ready_no_dr"
PCA_DIR = PROJECT_DIR / "model_ready_with_pca"
OUTPUT_DIR = PROJECT_DIR / "model_outputs" / "baseline_models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

target_col = "target_energy_next_60s_wh"
id_cols = ["log_id", "dataset_label"]

print("Output directory:", OUTPUT_DIR)

XGBoost is not installed. Install using: pip install xgboost
Output directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models


## 2. Helper functions

In [2]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def get_feature_cols(df, target_col, id_cols):
    return [
        col for col in df.columns
        if col not in id_cols + [target_col]
    ]


def evaluate_model(model_name, dataset_name, model, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\nTraining {model_name} on {dataset_name}...")

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    results = {
        "dataset": dataset_name,
        "model": model_name,
        "val_MAE": mean_absolute_error(y_val, val_pred),
        "val_RMSE": rmse(y_val, val_pred),
        "test_MAE": mean_absolute_error(y_test, test_pred),
        "test_RMSE": rmse(y_test, test_pred)
    }

    print(results)

    return results, model

## 3. Load non-PCA data

In [3]:
train_no_dr = pd.read_csv(NO_DR_DIR / "train_no_dr_imputed.csv")
val_no_dr = pd.read_csv(NO_DR_DIR / "val_no_dr_imputed.csv")
test_no_dr = pd.read_csv(NO_DR_DIR / "test_no_dr_imputed.csv")

feature_cols_no_dr = get_feature_cols(train_no_dr, target_col, id_cols)

X_train_no_dr = train_no_dr[feature_cols_no_dr]
X_val_no_dr = val_no_dr[feature_cols_no_dr]
X_test_no_dr = test_no_dr[feature_cols_no_dr]

y_train_no_dr = train_no_dr[target_col]
y_val_no_dr = val_no_dr[target_col]
y_test_no_dr = test_no_dr[target_col]

print("Non-PCA train shape:", X_train_no_dr.shape)
print("Non-PCA features:", feature_cols_no_dr)

Non-PCA train shape: (694549, 15)
Non-PCA features: ['battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_cell_v[0]', 'battery_status__voltage_cell_v[1]', 'battery_status__voltage_cell_v[2]', 'battery_status__voltage_cell_v[3]', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'power_w', 'time_s']
